<a href="https://colab.research.google.com/github/thuynhungp068-design/cuda_learning_with_google_colab/blob/main/cuda_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
%%writefile hello.cu

#include <cstdio>

__global__ void hello_kernel(){
  printf("Hello from GPU! block=%d,thread=%d\n",blockIdx.x,threadIdx.x);
}

int main(){
  hello_kernel<<<2,4>>>();
  cudaDeviceSynchronize();
  return 0;
}

Overwriting hello.cu


In [10]:
!nvcc -o hello hello.cu

In [12]:
!./hello

Hello from GPU! block=0,thread=0
Hello from GPU! block=0,thread=1
Hello from GPU! block=0,thread=2
Hello from GPU! block=0,thread=3
Hello from GPU! block=1,thread=0
Hello from GPU! block=1,thread=1
Hello from GPU! block=1,thread=2
Hello from GPU! block=1,thread=3


In [24]:
%%writefile vector_add.cu

#include <cstdio>

__global__ void vector_add(const float* a,const float* b,float* result,int n){
  int i=blockIdx.x*blockDim.x+threadIdx.x;
  if(i<n){
    result[i]=a[i]+b[i];
  }
}

int main(){
  const int N=1024;
  const size_t size=N*sizeof(float);

  float h_a[N],h_b[N],h_result[N];
  for(int i=0;i<N;i++){
    h_a[i]=(float)i;
    h_b[i]=(float)N-i;
  }

  float *d_a,*d_b,*d_result;
  cudaMalloc(&d_a,size);
  cudaMalloc(&d_b,size);
  cudaMalloc(&d_result,size);

  cudaMemcpy(d_a,h_a,size,cudaMemcpyHostToDevice);
  cudaMemcpy(d_b,h_b,size,cudaMemcpyHostToDevice);

  int threads_per_block=256;
  int blocks=(N+threads_per_block-1)/threads_per_block;
  vector_add<<<blocks,threads_per_block>>>(d_a,d_b,d_result,N);

  cudaDeviceSynchronize();

  cudaMemcpy(h_result,d_result,size,cudaMemcpyDeviceToHost);

  bool correct=true;
  for(int i=0;i<N;i++){
    if(h_result[i]!=N){
      correct=false;
      printf("Error at i=%d: %f\\n",i,h_result[i]);
      break;
    }
  }
  printf("%s\\n", correct ? "Vector add PASSED!" : "Vector add FAILED!");

  cudaFree(d_a);
  cudaFree(d_b);
  cudaFree(d_result);
  return 0;
}

Overwriting vector_add.cu


In [25]:
!nvcc -arch=sm_75 -o vector_add vector_add.cu

In [26]:
!./vector_add


Vector add PASSED!\n